# ANLP Project: Learned Controllers for Latent-Preserving Context Switching
### Google Colab / Kaggle 1-Click GPU Runner (Weeks 1 to 4)
This notebook runs the full pipeline on free cloud GPUs (e.g. NVIDIA T4 / P100) and automatically logs charts to Weights & Biases (W&B) and saves model checkpoints to Hugging Face Hub.

## 1. Setup & Dependencies

In [1]:
# Check GPU
!nvidia-smi

# Install requirements
!pip install -q torch transformers accelerate sentencepiece wandb huggingface_hub python-dotenv tqdm

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


## 2. Configure Credentials (W&B and Hugging Face)
Enter your free API keys below. If you don't have them yet, you can leave them blank and results will still save locally.

In [ ]:
import os
from getpass import getpass

# Optional: Set W&B key
wandb_key = getpass("Enter W&B API Key (or press enter to skip): ")
if wandb_key.strip():
    os.environ["WANDB_API_KEY"] = wandb_key.strip()
    os.environ["WANDB_PROJECT"] = "anlp-latent-context-switch"

# Optional: Set Hugging Face token
hf_token = getpass("Enter Hugging Face Token (or press enter to skip): ")
if hf_token.strip():
    os.environ["HF_TOKEN"] = hf_token.strip()
    os.environ["HF_REPO_ID"] = input("Enter HF Repo ID (e.g. username/anlp-controller): ").strip()

print("Credentials configured!")

## 3. Clone Repository or Upload Files
If running from a GitHub clone, navigate into the repo folder. Otherwise ensure `src/` and `data/` are in the working directory.

In [ ]:
# Example if cloning from github:
# !git clone https://github.com/<your-username>/ANLP_grp.git
# %cd ANLP_grp

# Verify topics file exists
!ls data/

## 4. Run the Full Week 1-4 Pipeline
This executes:
1. **Week 2**: Evaluates Agent C detection on the Naive Baseline.
2. **Week 3**: Instantiates and verifies the LatentSwitchController.
3. **Week 4**: Generates synthetic dialogues and pretrains Agent B's decoder head.

In [ ]:
# Use 1.5B model on GPU (or 0.5B for lightning speed)
!python run_pipeline.py \
    --mode all \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --device cuda \
    --baseline_samples 20 \
    --pretrain_samples 10 \
    --pretrain_epochs 10 \
    --batch_size 16

## 5. Inspect Results & Checkpoints

In [ ]:
!ls -lh checkpoints/
!ls -lh results/

import json
with open("results/naive_baseline_results.json") as f:
    baseline = json.load(f)
print(f"Naive Baseline Agent C Accuracy: {baseline['c_baseline_accuracy'] * 100:.1f}%")

with open("results/b_decoder_pretraining_results.json") as f:
    pretrain = json.load(f)
print(f"Pretrained B Decoder Best Accuracy: {pretrain['best_val_accuracy'] * 100:.1f}%")